In [1]:
import torch
import myneural as mn
import torch.nn.functional as F
import time
import pyarrow.parquet as pq
import sentencepiece as spm
import math
torch.manual_seed(42)

In [2]:
# training tokenizer --------------
TRAIN = False
if TRAIN:
    pf = pq.ParquetFile('data/tinystories0.parquet')
    with open('data/input.txt', 'w', encoding='utf-8') as f:
        for i in range(100):
            text_list = pf.read_row_group(i).column('text').to_pylist() # 1000 rows each row group     
            for story in text_list:
                f.write(story)
                
    spm.SentencePieceTrainer.train( # pyright: ignore[reportAttributeAccessIssue]
        input = 'data/input.txt',
        model_prefix = 'models/token_model',
        vocab_size = 4096,
        model_type = 'bpe',
        character_coverage = 1.0,
        byte_fallback = True,
        add_dummy_prefix = True,
        )

In [3]:
sp = spm.SentencePieceProcessor(model_file="models/token_model.model") # pyright: ignore[reportCallIssue]
sp.Encode("Lily found a needle.") # pyright: ignore[reportAttributeAccessIssue]
print(''.join(sp.IdToPiece(id) for id in sp.Encode("Bob saw a shiny golden penny.\n He liked it.") + [sp.eos_id()] + sp.Encode("Lily found a flower.")).replace('▁', ' ').replace('</s>', '</s>\n')) # EOS IDs are hidden in general.

 Bob saw a shiny golden penny. He liked it.</s>
 Lily found a flower.


In [4]:
config = mn.Config()

In [5]:
config.n_embd = 256
config.n_head = 8
config.n_layer = 8
config.mini_batch_size = 64 # B
config.context_length = 256 # T
config.total_batch_size = 1 * config.mini_batch_size * config.context_length # MUST BE DIVISBLE by (B*T)
config.device = 'cuda'
config.vocab_size = 4096

max_iter = 6000
B = config.mini_batch_size
T = config.context_length
grad_accum_steps = config.total_batch_size // (B*T)
assert config.total_batch_size % (B * T) == 0, "Total batch size must be divisble by (B*T)"

In [6]:
import dataset
set = dataset.ParquetDataset([
    'data/tinystories0.parquet',
    ],
    B = config.mini_batch_size,
    T= config.context_length,
    tokenizer=sp,
    append_eos=True,
    loop_around=True,
    shuffle_rgs=True,
    seed=42,
)

dataloader = torch.utils.data.DataLoader(
    set,
    batch_size=None,# batch_size = None because i already give (B, T) in the dataset.
    # if i gave a batch_size, it would give me (batch_size, B, T)
    num_workers=6,
    pin_memory=True,
    prefetch_factor=4, # 2 batches will be loaded ahead of time.
    persistent_workers=True
) 

In [7]:
val_set = dataset.ParquetDataset([
    'data/tinystories0val.parquet',
    ],
    B = config.mini_batch_size,
    T= config.context_length,
    tokenizer=sp,
    append_eos=True,
    loop_around=False,
    shuffle_rgs=False,
    seed=42,
)

val_dataloader = torch.utils.data.DataLoader(
    val_set,
    batch_size=None,
    num_workers=0,
    pin_memory=True,
) 

In [8]:
def get_lr(step):
    max_lr = 1e-3
    min_lr = 1e-4
    warmup_steps=200
    max_steps=6000
    if step < warmup_steps:
        return max_lr * (step+1)/warmup_steps 
    if step > max_steps:
        return min_lr
    decay_ratio = (step-warmup_steps)/(max_steps-warmup_steps)
    coefficient = 0.5 * (1.0 + math.cos(decay_ratio * math.pi)) # starts at 1 reduces to 0
    return min_lr + coefficient * (max_lr - min_lr)
    

In [9]:
torch.set_float32_matmul_precision('high')
model = mn.Transformer(config, dtype=torch.float32, device=config.device)
params = model.parameters()

In [10]:
optimizer_adamw = model.configure_optimizer(weight_decay=0.1, learning_rate=get_lr(0), device=config.device)
# optimizer_muon = model.configure_muon(weight_decay=0.1, learning_rate=get_lr(0), device=config.device)

In [11]:
# # Load the list of tensors
# loaded_params = torch.load("models/params_new.pth")

# # Manually re-assign them to your model
# for p, loaded_p in zip(model.parameters(), loaded_params):
#     p.data.copy_(loaded_p.data)

In [12]:
model_optim = torch.compile(model)
step = 0

In [13]:
def get_val_loss():
    max_batches = 25
    val_loss = []
    with torch.no_grad():
        for i, (xs, ys) in enumerate(val_dataloader):      
            if i >= max_batches:
                break
            xs = xs.to(device=config.device, non_blocking=True)
            ys= ys.to(device=config.device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                _, loss = model_optim(xs, targets=ys)
            val_loss.append(loss.item())
    return sum(val_loss)/len(val_loss)

In [14]:
data_iter = iter(dataloader)
lossi = []
timei = []
mean_stat_batches = 100
print_batches = 100
val_batches = 300
for _ in range(step, max_iter):
    t_start = time.time()
    optimizer_adamw.zero_grad(set_to_none=True)
    # optimizer_muon.zero_grad(set_to_none=True)
    # forward pass
    loss_accum = torch.tensor(0.0, device=config.device)
    for micro_step in range(grad_accum_steps):
        xs, ys= next(data_iter) # loop_around=True so we don't need to worry about StopIteration exception
        xs = xs.to(device=config.device, non_blocking=True)
        ys= ys.to(device=config.device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits, loss = model_optim(xs, targets=ys)
        loss = loss / grad_accum_steps # normalizer to avoid loss gradients being exagerrated by grad_accum_steps times.
        loss_accum += loss.detach()
        # backward pass
        loss.backward()
    
    norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
    
    # update
    lr = get_lr(step)
    for param_group in optimizer_adamw.param_groups:
        param_group['lr'] = lr
    optimizer_adamw.step()
    
    # for param_group in optimizer_muon.param_groups:
    #     param_group['lr'] = lr
    # optimizer_muon.step()
    
    torch.cuda.synchronize()
    t_end = time.time()
    
    # track stats    
    lossi.append(loss_accum.item())
    timei.append(t_end-t_start)
    if step % print_batches == 0:
        lossi = lossi[-mean_stat_batches:]
        timei = timei[-mean_stat_batches:]
        train_loss = sum(lossi)/len(lossi)
        if step % val_batches == 0:
            val_loss = get_val_loss()
        else: 
            val_loss = 0.0
        dt = sum(timei)/len(timei)
        tokens_per_sec = (xs.shape[0] * xs.shape[1] * grad_accum_steps)/dt # (B*T)/dt
        print(f"Step {step}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | dt: {dt*1000:.2f}ms | {int(tokens_per_sec):,d} tok/s")

    step += 1

W0123 01:53:38.901000 30200 site-packages\torch\_inductor\utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode


Step 0: Train Loss: 8.8200 | Val Loss: 8.8412 | dt: 19323.30ms | 847 tok/s
Step 100: Train Loss: 6.4692 | Val Loss: 0.0000 | dt: 66.01ms | 248,187 tok/s
Step 200: Train Loss: 4.3229 | Val Loss: 0.0000 | dt: 66.97ms | 244,663 tok/s
Step 300: Train Loss: 3.5806 | Val Loss: 3.3576 | dt: 65.35ms | 250,729 tok/s
Step 400: Train Loss: 3.1195 | Val Loss: 0.0000 | dt: 63.85ms | 256,583 tok/s
Step 500: Train Loss: 2.9824 | Val Loss: 0.0000 | dt: 63.77ms | 256,927 tok/s
Step 600: Train Loss: 2.7813 | Val Loss: 2.7698 | dt: 64.48ms | 254,113 tok/s
Step 700: Train Loss: 2.7291 | Val Loss: 0.0000 | dt: 64.33ms | 254,701 tok/s
Step 800: Train Loss: 2.5771 | Val Loss: 0.0000 | dt: 63.41ms | 258,402 tok/s
Step 900: Train Loss: 2.5320 | Val Loss: 2.5162 | dt: 63.57ms | 257,717 tok/s
Step 1000: Train Loss: 2.4911 | Val Loss: 0.0000 | dt: 69.79ms | 234,774 tok/s
Step 1100: Train Loss: 2.4387 | Val Loss: 0.0000 | dt: 69.85ms | 234,547 tok/s
Step 1200: Train Loss: 2.3306 | Val Loss: 2.3578 | dt: 68.92ms | 

In [18]:
val_loss = get_val_loss()
val_loss

1.8219066190719604

In [16]:
SAVE = False
if SAVE:
    params_list = list(model.parameters())
    torch.save(params_list, "models/params_muon.pth")

In [20]:
print(mn.prompt("Tim loves to eat toast. So he asked his mom to cook some", model=model, tokenizer=sp, new_tokens=256, debug=True, device=config.device))

</s>
 Tim loves to eat toast. So he asked his mom to cook some more jam on the stove. His mom said, "You should read cows to be good, Tim." Tim quietly put on his toast, and sets the chair behind his back. He pebbles on the big bowl and pretends it is fuel for the itch. With his mom's help, they scrubber the bowl and put it back in the bowl. It was very hot now! Tim closed his eyes and tried to keep the inside, but it was stuck. He asked his mom, "Mom, why is such the m sandwiches falling from the consequences?" His mom smiled and said, "Sometimes the other food that it is stuck outside on the stove, but with shopping things." Tim nodded. He knew that he would be humble just like struggling to find the food rain him when he read.</s>
 Once upon a time there were two friends, Jack and Fred. They were playing in the garden. They were exploring when suddenly Bob heard that they were confused. He said to Fred, "Coming Prunes!" Fred replied, "Watch me, I'm going to please!" Fred smiled and 